# Crypto News & Market Tools — Demo Notebook

Demonstrates each tool in `src/tools/` exactly as it will be called by the LangGraph agents: `fetch_rss_news`, `search_crypto_news`, `get_coin_market_snapshot`, `get_stock_quote`.

All tool logic lives in the `.py` modules under `src/tools/` — this notebook only imports and calls them.

**Setup**: copy `.env.example` to `.env` (in the project root) and fill in `SERPAPI_KEY` / `ALPHAVANTAGE_KEY` before running the SerpAPI and AlphaVantage sections.

In [1]:
import logging
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from dotenv import load_dotenv

load_dotenv(PROJECT_ROOT / ".env")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(name)s %(levelname)s %(message)s",
)

from src.tools import (
    fetch_rss_news,
    search_crypto_news,
    get_coin_market_snapshot,
    get_stock_quote,
)

## 1. `fetch_rss_news` — RSS feeds (CoinDesk, Decrypt)

In [2]:
rss_result = fetch_rss_news.invoke({"hours_back": 48})

print(f"count: {rss_result.get('count')}")
print(rss_result["articles"][0] if rss_result.get("articles") else rss_result)

assert "error" not in rss_result, rss_result.get("error")
assert rss_result["count"] > 0, "Expected at least one RSS article"

2026-07-16 11:12:30,845 src.tools.cache INFO Cache miss for fetch_rss_news; calling underlying function


count: 45
{'title': 'Live updates: ZachXBT calls hardware wallets complete garbage; BTC steady after Korea rate hike', 'url': 'https://www.coindesk.com/markets/2026/07/16/live-updates-zachxbt-calls-hardware-wallets-complete-garbage-btc-steady-near-usd65-000', 'published_at': '2026-07-16T06:20:00+00:00', 'source': 'CoinDesk: Bitcoin, Ethereum, Crypto News and Price Data', 'summary_raw': ''}


## 2. `search_crypto_news` — SerpAPI Google News search

In [3]:
serp_result = search_crypto_news.invoke({"query": "Bitcoin ETF", "num_results": 5})

print(f"count: {serp_result.get('count')}")
print(serp_result["articles"][0] if serp_result.get("articles") else serp_result)

if "error" in serp_result:
    print("NOTE: set SERPAPI_KEY in .env to run this section successfully.")
else:
    assert serp_result["count"] > 0, "Expected at least one search result"

2026-07-16 11:12:31,919 src.tools.cache INFO Cache miss for search_crypto_news; calling underlying function


count: 5
{'title': "Bitcoin ETFs Are Seeing Massive Outflows, but the Price of HYPE Keeps Rising. Here's Why I'm Bullish on Hyperliquid.", 'url': 'https://finance.yahoo.com/markets/crypto/articles/bitcoin-etfs-seeing-massive-outflows-100000757.html', 'published_at': '2026-07-15T10:00:00+00:00', 'source': 'Yahoo Finance', 'summary_raw': ''}


## 3. `get_coin_market_snapshot` — CoinGecko market data

In [4]:
market_result = get_coin_market_snapshot.invoke({"coin_ids": ["bitcoin", "ethereum"]})

print(market_result)

assert "error" not in market_result, market_result.get("error")
assert market_result["count"] == 2, "Expected snapshot data for both coins"

2026-07-16 11:12:34,990 src.tools.cache INFO Cache miss for get_coin_market_snapshot; calling underlying function


{'coins': [{'id': 'bitcoin', 'symbol': 'btc', 'name': 'Bitcoin', 'current_price': 64094, 'price_change_percentage_24h': -1.06837, 'total_volume': 30758565867}, {'id': 'ethereum', 'symbol': 'eth', 'name': 'Ethereum', 'current_price': 1885.47, 'price_change_percentage_24h': 0.15709, 'total_volume': 13098738789}], 'count': 2}


## 4. `get_stock_quote` — AlphaVantage equity quote

In [5]:
stock_result = get_stock_quote.invoke({"symbol": "COIN"})

print(stock_result)

if "error" in stock_result:
    print("NOTE: set ALPHAVANTAGE_KEY in .env to run this section successfully.")
else:
    assert stock_result["price"] > 0, "Expected a positive stock price"

2026-07-16 11:12:35,480 src.tools.cache INFO Cache miss for get_stock_quote; calling underlying function


{'symbol': 'COIN', 'price': 167.21, 'change': 5.71, 'change_percent': '3.5356%', 'latest_trading_day': '2026-07-15'}


## 5. Caching demo

Call the same tool twice with identical arguments and confirm the second call is served from the disk cache instead of hitting the real API — shown both via the `cache.py` INFO-level log lines above ("Cache hit" vs "Cache miss") and via timing.

In [6]:
import time

t0 = time.perf_counter()
first_call = get_coin_market_snapshot.invoke({"coin_ids": ["bitcoin"]})
t1 = time.perf_counter()
second_call = get_coin_market_snapshot.invoke({"coin_ids": ["bitcoin"]})
t2 = time.perf_counter()

first_duration = t1 - t0
second_duration = t2 - t1

print(f"First call:  {first_duration:.4f}s")
print(f"Second call: {second_duration:.4f}s (should be much faster — served from data/cache/)")

assert first_call == second_call, "Cached result should be identical to the original"
assert second_duration < first_duration, "Cached call should be faster than the real API call"

2026-07-16 11:12:35,866 src.tools.cache INFO Cache miss for get_coin_market_snapshot; calling underlying function


2026-07-16 11:12:36,220 src.tools.cache INFO Cache hit for get_coin_market_snapshot (age=0.0s, ttl=1.0h)


First call:  0.3527s
Second call: 0.0029s (should be much faster — served from data/cache/)
